In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Update this if your model checkpoint corresponds to a specific data
# fraction from the learning-curve experiment (e.g. "Small Llama (100% data)")
MODEL_LABEL = "Small Llama (custom-trained)"

DATA_DIR = Path("/content/drive/MyDrive/Thesis/results/small_llama")
OUTPUT_DIR = Path("/content/drive/MyDrive/Thesis/results/small_llama_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Load all result files
with open(DATA_DIR / "overall_summary.json", encoding='utf-8') as f:
    overall = json.load(f)

with open(DATA_DIR / "stability_results.json", encoding='utf-8') as f:
    stability = json.load(f)

phenomena = list(overall["phenomenon_accuracies"].keys())
detailed = {}
for phen in phenomena:
    with open(DATA_DIR / f"detailed_{phen}.json", encoding='utf-8') as f:
        detailed[phen] = json.load(f)

print(f"Model: {MODEL_LABEL}")
print(f"Overall accuracy: {overall['overall_accuracy']:.2%} "
      f"({overall['total_correct_pairs']}/{overall['total_evaluated_pairs']})")
print(f"\nPhenomena loaded: {len(phenomena)}")
for p in phenomena:
    print(f"  {p}: {detailed[p]['correct']}/{detailed[p]['total']} pairs")

## Table 1: Accuracy by Phenomenon

In [ ]:
rows = []
for phen, acc in overall["phenomenon_accuracies"].items():
    rows.append({
        "Phenomenon": phen,
        "Accuracy (%)": round(acc * 100, 1),
        "Correct": detailed[phen]["correct"],
        "Total": detailed[phen]["total"],
    })
df_acc = pd.DataFrame(rows).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)

overall_row = pd.DataFrame([{
    "Phenomenon": "OVERALL",
    "Accuracy (%)": round(overall["overall_accuracy"] * 100, 1),
    "Correct": overall["total_correct_pairs"],
    "Total": overall["total_evaluated_pairs"],
}])
df_acc_full = pd.concat([df_acc, overall_row], ignore_index=True)

print(df_acc_full.to_string(index=False))
df_acc_full.to_csv(OUTPUT_DIR / "table1_accuracy_by_phenomenon.csv", index=False)
print(f"\n\u2713 Saved to {OUTPUT_DIR / 'table1_accuracy_by_phenomenon.csv'}")

## Chart 1: Accuracy by Phenomenon

In [ ]:
df_plot = pd.DataFrame([
    {"Phenomenon": p, "Accuracy": a * 100} for p, a in overall["phenomenon_accuracies"].items()
]).sort_values("Accuracy", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = plt.cm.RdYlGn(df_plot["Accuracy"].values / 100)
bars = ax.bar(df_plot["Phenomenon"], df_plot["Accuracy"], color=colors, edgecolor='white')
ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='Chance (50%)')
for bar, val in zip(bars, df_plot["Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5, f"{val:.0f}%", ha='center', fontsize=10)
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"{MODEL_LABEL} \u2014 Accuracy by Phenomenon")
ax.set_ylim(0, 105)
ax.legend()
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "chart1_accuracy_by_phenomenon.png", bbox_inches='tight')
plt.show()

## Chart 2 + Table 2: Sample-Size Stability

In [ ]:
sizes = [25, 50, 75, 100]

fig, ax = plt.subplots(figsize=(9, 5.5))
for phen in phenomena:
    means = [stability[phen][str(s)]["mean_accuracy"] * 100 for s in sizes]
    mins = [stability[phen][str(s)]["min_accuracy"] * 100 for s in sizes]
    maxs = [stability[phen][str(s)]["max_accuracy"] * 100 for s in sizes]
    line, = ax.plot(sizes, means, marker='o', label=phen, linewidth=1.8)

ax.set_xlabel("Sample size (n pairs)")
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"{MODEL_LABEL} \u2014 Accuracy Stability Across Sample Sizes\n(shaded band = min\u2013max across 5 random draws)")
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "chart2_sample_size_stability.png", bbox_inches='tight')
plt.show()

In [ ]:
# MPAD (mean pairwise absolute deviation) per phenomenon — how much the
# accuracy estimate wobbles across sample sizes (lower = more stable)
stab_rows = []
for phen in phenomena:
    row = {"Phenomenon": phen}
    size_means = {}
    for s in sizes:
        m = stability[phen][str(s)]["mean_accuracy"] * 100
        row[f"n={s} mean"] = round(m, 1)
        row[f"n={s} range"] = (f"{stability[phen][str(s)]['min_accuracy']*100:.0f}"
                                 f"-{stability[phen][str(s)]['max_accuracy']*100:.0f}")
        size_means[s] = m
    diffs = [abs(size_means[a] - size_means[b]) for a, b in combinations(sizes, 2)]
    row["MPAD (pp)"] = round(np.mean(diffs), 2)
    stab_rows.append(row)

df_stab = pd.DataFrame(stab_rows)
print(df_stab.to_string(index=False))
print(f"\nOverall mean MPAD across phenomena: {df_stab['MPAD (pp)'].mean():.2f} pp")

df_stab.to_csv(OUTPUT_DIR / "table2_stability_summary.csv", index=False)
print(f"\n\u2713 Saved to {OUTPUT_DIR / 'table2_stability_summary.csv'}")

## Confidence Analysis (ΔNLL)

ΔNLL = surprisal(ungrammatical) − surprisal(grammatical), computed per pair.
ΔNLL > 0 means the model favored the grammatical sentence.

In [ ]:
delta_rows = []
for phen in phenomena:
    for pair in detailed[phen]["pairs"]:
        gram_s = pair.get("gram_surprisal")
        ungram_s = pair.get("ungram_surprisal")
        if gram_s is not None and ungram_s is not None:
            delta_rows.append({
                "Phenomenon": phen,
                "delta_nll": ungram_s - gram_s,
                "correct": pair["correct"]
            })

delta_df = pd.DataFrame(delta_rows)
print(f"Loaded {len(delta_df):,} pair-level confidence values")
delta_df.head()

In [ ]:
# Box plot: per-phenomenon distribution of confidence
phen_order = delta_df.groupby("Phenomenon")["delta_nll"].mean().sort_values().index

fig, ax = plt.subplots(figsize=(9, 5))
box_data = [delta_df[delta_df["Phenomenon"] == p]["delta_nll"].values for p in phen_order]
ax.boxplot(box_data, vert=False, labels=phen_order, showfliers=True)
ax.axvline(0, color='blue', linestyle='--', linewidth=1)
ax.set_xlabel("\u0394NLL (higher is better \u2014 model favors grammatical sentence)")
ax.set_title(f"{MODEL_LABEL} \u2014 Per-Phenomenon Confidence Distribution")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "chart3_delta_nll_boxplot.png", bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: average confidence vs. accuracy per phenomenon
avg_delta = delta_df.groupby("Phenomenon")["delta_nll"].mean()
avg_acc = pd.Series(overall["phenomenon_accuracies"])

merged = pd.DataFrame({"avg_delta_nll": avg_delta, "avg_accuracy": avg_acc}).dropna()

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(merged["avg_delta_nll"], merged["avg_accuracy"] * 100, s=60)
for phen, row in merged.iterrows():
    ax.annotate(phen, (row["avg_delta_nll"], row["avg_accuracy"] * 100),
                fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel("Average \u0394NLL")
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"{MODEL_LABEL} \u2014 Confidence vs. Accuracy per Phenomenon")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "chart4_confidence_vs_accuracy.png", bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print(f"{MODEL_LABEL}")
print(f"Overall accuracy: {overall['overall_accuracy']:.1%} "
      f"({overall['total_correct_pairs']}/{overall['total_evaluated_pairs']})\n")

print("Easiest phenomena:")
print(df_acc.head(2).to_string(index=False))

print("\nHardest phenomena:")
print(df_acc.tail(2).to_string(index=False))

below_chance = df_acc[df_acc["Accuracy (%)"] < 50]
if len(below_chance) > 0:
    print(f"\n\u26a0 Below chance (50%):")
    print(below_chance.to_string(index=False))
else:
    print("\n\u2713 No phenomena below chance.")

print(f"\nAll outputs saved to: {OUTPUT_DIR}")